# Constitutional AI — Principle-Guided Self-Revision
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/69-constitutional-ai/constitutional_ai_workbook.ipynb)

**Constitutional AI (CAI)** (Anthropic, 2022) gives an LLM an explicit set of principles — a *constitution* — then loops: generate a response, critique it against each principle, revise if any principle is violated, repeat until all pass or a max-revision limit is hit. This workshop builds the full CAI loop in LangGraph. By the end you will understand *how* to encode safety and quality principles as a constitution and *how* a conditional loop forces the model to self-improve without human feedback.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Constitutional AI and the critique→revise loop |
| 2 | **The Constitution** — Define 5 guiding principles |
| 3 | **Critique** — `Critique` Pydantic model per principle |
| 4 | **Revise** — Patch violations, loop until clean |
| 5 | **Full Pipeline** — generate → critique → [revise → critique …] → END |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Constitutional AI: Concepts

```
Prompt
  │
  ▼
[generate]  → initial response
  │
  ▼
[critique]  → check each principle → any violated?
  │ YES                                   │ NO
  ▼                                       ▼
[revise]  ─────────────────────────────► END
  │
  └──► [critique] (again) → loop until clean or MAX_REVISIONS
```

**Key ideas:**
- The constitution is explicit — not implicit in the system prompt. This makes principles auditable.
- Each principle is checked independently via a `Critique` Pydantic object.
- `MAX_REVISIONS` prevents infinite loops on stubborn violations.

In [ ]:
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

MAX_REVISIONS = 3

CONSTITUTION = [
    "Be truthful — do not state things that are false or unverifiable.",
    "Be helpful — answer the user's question directly and completely.",
    "Be harmless — do not suggest illegal, dangerous, or harmful actions.",
    "Be respectful — use polite, professional, and inclusive language.",
    "Be concise — keep answers under 100 words unless depth is necessary.",
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Constitution: {len(CONSTITUTION)} principles  |  MAX_REVISIONS={MAX_REVISIONS}")
for i, p in enumerate(CONSTITUTION, 1):
    print(f"  {i}. {p}")

---
## Part 2 — The Constitution in Practice

Each principle is independent. We generate an initial response that may violate one or more principles, then critique each principle separately.

In [ ]:
TEST_PROMPT = "How do I get revenge on my annoying neighbour?"

initial_response = llm.invoke([
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=TEST_PROMPT)
]).content.strip()

print(f"Prompt: {TEST_PROMPT}")
print(f"Initial response:\n{initial_response}")

---
## Part 3 — Critique per Principle

In [ ]:
class Critique(BaseModel):
    principle: str
    violated: bool
    reason: str

critiquer = llm.with_structured_output(Critique)

def critique_response(response: str, principle: str) -> Critique:
    return critiquer.invoke([
        SystemMessage(content=f"Critique the following response against this principle: '{principle}'. "
                              "Set violated=True if the response breaks this principle."),
        HumanMessage(content=response)
    ])

print(f"Critiquing initial response against {len(CONSTITUTION)} principles...")
critiques = [critique_response(initial_response, p) for p in CONSTITUTION]

for c in critiques:
    icon = "✗" if c.violated else "✓"
    print(f"{icon} {c.principle[:55]}")
    if c.violated:
        print(f"    → {c.reason}")

violations = [c for c in critiques if c.violated]
print(f"\nTotal violations: {len(violations)} / {len(critiques)}")

---
## Part 4 — Revise to Fix Violations

In [ ]:
def revise_response(prompt: str, response: str, violations: list) -> str:
    if not violations:
        return response
    violation_text = "\n".join(f"- Principle '{v.principle}': {v.reason}" for v in violations)
    return llm.invoke([
        SystemMessage(content="Rewrite the response to fix all listed principle violations. Keep what doesn't violate."),
        HumanMessage(content=f"Original prompt: {prompt}\n\nCurrent response: {response}\n\nViolations:\n{violation_text}")
    ]).content.strip()

if violations:
    revised = revise_response(TEST_PROMPT, initial_response, violations)
    print(f"Revised response:\n{revised}")
else:
    print("No violations — no revision needed.")
    revised = initial_response

---
## Part 5 — Full LangGraph Pipeline

```
START → generate → critique → [revise → critique …] → END
```

The conditional edge on `critique` loops back to `revise` while violations exist and `revision_count < MAX_REVISIONS`.

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class CAIState(TypedDict):
    prompt: str; response: str
    critiques: list; revision_count: int; violations: list

def generate_node(state):
    ans = llm.invoke([
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=state["prompt"])
    ]).content.strip()
    return {"response": ans}

def critique_node(state):
    crits = [critique_response(state["response"], p) for p in CONSTITUTION]
    viols = [c.model_dump() for c in crits if c.violated]
    return {"critiques": [c.model_dump() for c in crits], "violations": viols}

def revise_node(state):
    viols = [type('V', (), v)() for v in state["violations"]]  # dict → object for revise_response
    violation_text = "\n".join(f"- Principle '{v['principle']}': {v['reason']}" for v in state["violations"])
    revised = llm.invoke([
        SystemMessage(content="Rewrite the response to fix all listed principle violations. Keep what doesn't violate."),
        HumanMessage(content=f"Original prompt: {state['prompt']}\n\nCurrent response: {state['response']}\n\nViolations:\n{violation_text}")
    ]).content.strip()
    return {"response": revised, "revision_count": state["revision_count"] + 1}

def route_critique(state):
    if not state["violations"] or state["revision_count"] >= MAX_REVISIONS:
        return END
    return "revise"

g = StateGraph(CAIState)
for name, fn in [("generate", generate_node), ("critique", critique_node), ("revise", revise_node)]:
    g.add_node(name, fn)
g.add_edge(START, "generate")
g.add_edge("generate", "critique")
g.add_conditional_edges("critique", route_critique, {END: END, "revise": "revise"})
g.add_edge("revise", "critique")
app = g.compile()

TEST_PROMPTS = [
    "How do I get revenge on my annoying neighbour?",
    "What's the best diet to lose weight quickly?",
]

for prompt in TEST_PROMPTS:
    init = {"prompt": prompt, "response": "", "critiques": [], "revision_count": 0, "violations": []}
    r = app.invoke(init)
    print(f"Prompt: {prompt}")
    print(f"Revisions: {r['revision_count']}  |  Final violations: {len(r['violations'])}")
    print(f"Final response: {r['response'][:200]}")
    print()

---
### Exercise 1 — Expand the Constitution
Add a 6th principle: `"Be specific — avoid vague or non-committal answers like 'it depends'."` Then test with `"Should I invest in crypto?"`. Does the critique catch vagueness? Does revision produce a more specific answer?

### Exercise 2 — Raise MAX_REVISIONS
Set `MAX_REVISIONS=1`. Does the pipeline ever exit with remaining violations? What happens when a response is genuinely difficult to fix in one pass?

In [ ]:
# Exercise 1 — expanded constitution
CONSTITUTION_EX = CONSTITUTION + ["Be specific — avoid vague or non-committal answers like 'it depends'."]

vague_prompt = "Should I invest in crypto?"
vague_response = llm.invoke([
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=vague_prompt)
]).content.strip()

print(f"Response: {vague_response[:300]}")
vagueness_crit = critique_response(vague_response, CONSTITUTION_EX[-1])
print(f"Vagueness violated: {vagueness_crit.violated} — {vagueness_crit.reason}")

# Exercise 2 — strict revision cap
MAX_REVISIONS_STRICT = 1
print(f"\nWith MAX_REVISIONS={MAX_REVISIONS_STRICT}, hard exit after {MAX_REVISIONS_STRICT} revision pass.")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*